# Fonda pipeline runner

Runs `scripts/run_pipeline_stages.py` end to end: **data prep → SGD → MLP baseline →
experience replay → evaluation**. Every cell is safe to re-run in a brand-new
session -- clone-or-pull, dependency install, git config, and the dataset symlinks
are all idempotent. Just **Run All** each session; finished work is skipped
automatically and interrupted work resumes.

**One-time setup before your first run:**
1. Notebook Settings (right panel) -> **Internet: On**.
2. **Add Input** -> attach your `fonda-training-data` dataset.
3. Add-ons -> **Secrets** -> add a secret named exactly `GITHUB_TOKEN`, holding a
   GitHub Personal Access Token with **write** access to this repo (needed for
   `--git-push` -- without this, training results are lost when the session ends,
   since `/kaggle/working` doesn't survive a session boundary on its own).


In [ ]:
import pathlib
import subprocess

REPO_URL = "https://github.com/bartuturan/Online-Learning-for-Continuous-Forest-Monitoring-Addressing-Concept-Drift-in-Disturbance-Detection."
BRANCH = "pipeline-rerun"
ROOT = pathlib.Path("/kaggle/working/repo")

if (ROOT / ".git").exists():
    print("Repo already present -- pulling latest...")
    result = subprocess.run(["git", "pull"], cwd=ROOT, capture_output=True, text=True)
else:
    print("Cloning repo...")
    result = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)
result.check_returncode()


In [ ]:
!pip install -q "zarr>=3" "scikit-learn>=1.7"


In [ ]:
import inspect

import sklearn
import xarray as xr
import zarr
from sklearn.neural_network import MLPClassifier

print("zarr:", zarr.__version__)
print("scikit-learn:", sklearn.__version__)

engines = list(xr.backends.list_engines())
print("xarray engines:", engines)
assert "zarr" in engines, (
    "zarr isn't registered as an xarray backend -- restart the kernel (not just "
    "re-run cells) and re-run from the pip install cell above."
)

sig = inspect.signature(MLPClassifier.partial_fit)
print("MLPClassifier.partial_fit signature:", sig)
assert "sample_weight" in sig.parameters, (
    "scikit-learn is too old for sample_weight in partial_fit -- restart the "
    "kernel and re-run from the pip install cell above."
)
print("Dependency check OK.")


In [ ]:
import subprocess

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")  # Add-ons > Secrets, named exactly GITHUB_TOKEN

subprocess.run(["git", "config", "user.email", "kaggle-pipeline@users.noreply.github.com"], cwd=ROOT, check=True)
subprocess.run(["git", "config", "user.name", "Kaggle Pipeline Runner"], cwd=ROOT, check=True)

# Note: this repo's name unusually ends in a literal period -- match REPO_URL
# above exactly rather than appending ".git" (which would double it up).
push_url = f"https://{GITHUB_TOKEN}@github.com/bartuturan/Online-Learning-for-Continuous-Forest-Monitoring-Addressing-Concept-Drift-in-Disturbance-Detection."
subprocess.run(["git", "remote", "set-url", "origin", push_url], cwd=ROOT, check=True)
print("Git configured for push access (token not printed).")


In [ ]:
import pathlib

candidates = [
    pathlib.Path("/kaggle/input/fonda-training-data"),
    pathlib.Path("/kaggle/input/datasets/bartuturan/fonda-training-data"),
]
SRC = next((p for p in candidates if p.exists()), None)
if SRC is None:
    raise FileNotFoundError(
        f"fonda-training-data dataset not found under any of: {candidates}. "
        f"Attach it via 'Add Input' in the notebook's Data panel."
    )
print(f"Using dataset at: {SRC}")

for name in ("training_data_with_features_plus_monthly_indices.zarr", "data_split.npz"):
    link = ROOT / name
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(SRC / name, target_is_directory=name.endswith(".zarr"))
    print(f"  linked {name}")


In [ ]:
import os

os.environ["MLP_FEATURE_CACHE_DIR"] = "/kaggle/working/feature_cache_disk"
print("MLP_FEATURE_CACHE_DIR set.")


In [ ]:
%cd /kaggle/working/repo
!python -u scripts/run_pipeline_stages.py --list


### Check the status table above before running

Confirm it matches what you expect (which stages are DONE vs TODO). The run below
executes everything still outstanding, in order: **data prep → SGD → MLP baseline →
experience replay → evaluation**. Already-DONE stages are skipped instantly, so
this is also the command you re-run in a later session to continue where the last
one stopped.

The reservoir-sampling variants are excluded by default (they're not evaluated by
any of the 50 families in `src/eval/families.py`, so skipping them costs the
evaluation nothing). Run one explicitly with `--only <stage_id>` if you ever want
it back; `--list` above shows every stage id.

`--time-budget 8.0` stops cleanly before Kaggle's session limit rather than being
killed mid-notebook. `--git-push` commits and pushes `experiments/` after every
stage, so progress survives the session ending.


In [ ]:
!python -u scripts/run_pipeline_stages.py --time-budget 8.0 --git-push
